# Autodecodificadores Variacionales, Reparametrización y Auditoría del Espacio Latente

* **Modalidad:** Individual
* **Duración:** 2 horas y 30 minutos (150 minutos)
* **Entorno:** Python (PyTorch, NumPy, Matplotlib)

### Reglas y Restricciones de la Evaluación
1. **Prohibido usar datasets públicos** (MNIST, Fashion-MNIST, etc.). Debes generar tus propios datos sintéticos.
2. **Prohibido usar capas convolucionales** (`nn.Conv2d`). La arquitectura debe ser estrictamente densa (`nn.Linear`).
3. **Prohibido usar PCA o t-SNE** para visualizar el espacio latente. El cuello de botella debe tener exactamente 2 dimensiones.
4. Todos los parámetros aleatorios dependen de tu **ID de estudiante** (`SEED`).

---
## Material de Consulta Permitido

Durante las 2.5 horas de la práctica puedes consultar la documentación oficial de PyTorch, NumPy y Matplotlib, así como las siguientes guías y videos de referencia:

### Guías Escritas y Código de Referencia (PyTorch)
* [Tutorial de autoencoders variacionales: Explicación de los VAEs (Codecademy)](https://www.codecademy.com/article/variational-autoencoder-tutorial-vaes-explained) — Estructuración de capas lineales, divergencia KL y proyección latente en 2D.
* [Implementación de un VAE en PyTorch (Mikel GDA)](https://medium.com/@mikelgda/implementing-a-variational-autoencoder-in-pytorch-ddc0bb5ea1e7) — Separación de capas de media (`mu`) y log-varianza (`logvar`).
* [Inmersión profunda en VAEs con PyTorch (PyImageSearch)](https://pyimagesearch.com/2023/10/02/a-deep-dive-into-variational-autoencoders-with-pytorch/) — Equilibrio matemático de la función ELBO y efecto del hiperparámetro $\beta$.
* [Repositorio Pytorch-VAE-tutorial (GitHub)](https://github.com/Jackson-Kang/Pytorch-VAE-tutorial) — Ejemplo minimalista del VAE estándar de Kingma y Welling.

### Videos Explicativos (Truco de Reparametrización y Grafo Computacional)
* [What is the Reparameterization Trick? (YouTube)](https://www.youtube.com/watch?v=Y6R1ueen0J4) — Por qué el muestreo aleatorio rompe la diferenciabilidad y cómo aislar $\epsilon$.
* [Variational Autoencoder - Explained (YouTube)](https://www.youtube.com/watch?v=geH5HnRapRs) — Intuición sobre los vectores $\mu$, $\sigma$ y la ecuación $z = \mu + \sigma \odot \epsilon$.
* [The Reparameterization Trick in 60 Seconds (YouTube Shorts)](https://www.youtube.com/shorts/XI5laN51HQI) — Resumen rápido sobre la regla de la cadena y los nodos estocásticos.
* [Autoencoders: explicación y tutorial en Python (YouTube)](https://www.youtube.com/watch?v=aLcDJoG0pec) — Repaso en español sobre el cuello de botella (*bottleneck*) y reconstrucción.

---
## Paso 0: Configuración de Identidad (10 minutos)
Asigna los **últimos 3 dígitos de tu matrícula** a la variable `SEED`. No modifiques la fórmula de `BETA` ni la de `ANGLE`.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt

# TODO: Reemplaza 000 con los últimos 3 dígitos de tu matrícula
SEED = 000

assert isinstance(SEED, int) and 0 <= SEED <= 999, "SEED debe ser un entero entre 0 y 999."

np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Parámetros personalizados derivados de tu matrícula
BETA = 0.5 + (SEED % 10) * 0.15
ANGLE = SEED % 45

print(f"Configuración lista | SEED: {SEED} | BETA: {BETA:.2f} | ÁNGULO CRUZ: {ANGLE}° | Device: {device}")

---
## Parte 1: Generación del Dataset Sintético (25 minutos)
Crea una función `generar_dataset(n_samples=4000, seed=SEED)` que genere imágenes de **24 × 24 píxeles** en escala de grises:
* **2,000 elipses huecas** (grosor de borde ~2 píxeles, con radios variables dentro del lienzo).
* **2,000 cruces inclinadas** rotadas exactamente `ANGLE` (`SEED % 45`) grados.
* Agrega **ruido gaussiano** con desviación estándar $\sigma = 0.08$ y aplica `np.clip(..., 0.0, 1.0)`.
* Devuelve `X` con forma `(4000, 1, 24, 24)` o `(4000, 576)` en `float32`, y `y` con etiquetas `0` (elipse) y `1` (cruz).

In [ ]:
def generar_dataset(n_samples=4000, seed=SEED):
    rng = np.random.default_rng(seed)
    X = np.zeros((n_samples, 24, 24), dtype=np.float32)
    y = np.zeros(n_samples, dtype=np.int64)

    # TODO: Genera mitad elipses huecas (clase 0) y mitad cruces rotadas ANGLE grados (clase 1)
    # Sugerencia: Usa mallas de coordenadas np.meshgrid(np.arange(24), np.arange(24))

    # TODO: Agrega ruido gaussiano (std=0.08) y normaliza en [0, 1]

    return X, y

X_data, y_data = generar_dataset(4000, SEED)

# Verificación automática de dimensiones y rango
assert X_data.shape[0] == 4000 and X_data.size == 4000 * 576, "El dataset debe tener 4000 imágenes de 24x24."
assert X_data.min() >= 0.0 and X_data.max() <= 1.0, "Los píxeles deben estar normalizados entre 0 y 1."

# TODO: Grafica una cuadrícula de 4x4 ejemplos reales mezclados antes de entrenar

---
## Parte 2: Arquitectura del VAE y Truco de Reparametrización (45 minutos)
Implementa la clase `VAE` respetando estrictamente estas dimensiones:
* **Entrada:** `576` (24 × 24 aplanado)
* **Encoder:** `576 -> 256 -> 64` con activación `nn.LeakyReLU(0.2)`
* **Espacio Latente:** Dos capas lineales paralelas desde `64` hacia `2` (`fc_mu` y `fc_logvar`)
* **Decoder:** `2 -> 64 -> 256 -> 576` con `LeakyReLU(0.2)` en capas ocultas y `Sigmoid` en la salida
* **Pérdida ELBO:** Implementa `loss_function(recon_x, x, mu, logvar, beta=BETA)` usando Binary Cross Entropy (o MSE suma por muestra) + `beta * D_KL`.

In [ ]:
class VAE(nn.Module):
    def __init__(self):
        super(VAE, self).__init__()
        # TODO: Define las capas del Encoder (576 -> 256 -> 64)

        # TODO: Define las proyecciones latentes (64 -> 2 para mu y logvar)

        # TODO: Define las capas del Decoder (2 -> 64 -> 256 -> 576)

    def encode(self, x):
        # TODO: Retorna mu y logvar
        pass

    def reparameterize(self, mu, logvar):
        # TODO: Implementa z = mu + std * eps con ruido normal estándar en el mismo device
        pass

    def decode(self, z):
        # TODO: Reconstruye la imagen con salida Sigmoid
        pass

    def forward(self, x):
        mu, logvar = self.encode(x.view(-1, 576))
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

def vae_loss(recon_x, x, mu, logvar, beta=BETA):
    # TODO: Calcula pérdida de reconstrucción + beta * divergencia KL
    pass

---
## Parte 3: Entrenamiento y Sabotaje al Truco de Reparametrización (30 minutos)
1. Divide tus datos en **3,000 entrenamiento** y **1,000 prueba**.
2. Entrena tu `VAE` correcto durante **25 épocas** y grafica las curvas de pérdida (Reconstrucción y KL).
3. **El Sabotaje:** Crea la clase `VAESaboteado(VAE)` sobrescribiendo `reparameterize(self, mu, logvar)` para muestrear `z` usando `np.random.normal` o desconectando el grafo (`.detach()`). Entrénalo 10 épocas.

In [ ]:
# TODO: Prepara DataLoaders (3000 train / 1000 test)

# TODO: Entrena el modelo VAE correcto por 25 épocas y guarda historial de pérdidas

# TODO: Define VAESaboteado rompiendo el flujo de gradientes en reparameterize()
class VAESaboteado(VAE):
    def reparameterize(self, mu, logvar):
        # TODO: Rompe el truco de reparametrización aquí
        pass

# TODO: Ejecuta el entrenamiento del modelo saboteado por 10 épocas (captura el error o verifica los gradientes)

### Análisis Forense del Sabotaje (Máximo 120 palabras)
**Instrucción:** Explica qué ocurrió al ejecutar `VAESaboteado`, mencionando explícitamente los valores numéricos de pérdida/gradientes obtenidos en tu corrida y cómo afecta la **regla de la cadena** cuando existe un nodo estocástico sin reparametrizar dentro del grafo computacional.

> *Escribe tu respuesta aquí...*

---
## Parte 4: Exploración y Generación en el Espacio Latente (40 minutos)
Con tu modelo `VAE` correcto entrenado:
1. **Mapa del Espacio Oculto:** Pasa las **1,000 imágenes de prueba** por el encoder y grafica sus coordenadas latentes $(z_1, z_2)$ coloreadas según su clase real (Elipses vs. Cruces).
2. **Muestreo Generativo (Manifold Grid):** Genera una cuadrícula uniforme de $12 \times 12$ puntos en el rango $[-2.5, 2.5] \times [-2.5, 2.5]$, pásalos únicamente por `model.decode(z)` y visualiza la matriz completa de $12 \times 12$ imágenes generadas.

In [ ]:
# TODO: 1. Proyección 2D de las 1,000 muestras de prueba en el espacio latente (scatter plot)

# TODO: 2. Cuadrícula generativa de 12x12 en [-2.5, 2.5] x [-2.5, 2.5] decodificada